# SugarSense: Early Diabetes Risk Screening with Supervised Machine Learning
**Module**: Supervised Machine Learning Mini Project  
**Goal**: Predict diabetes risk from 8 clinical measurements, optimize for medical screening (Recall & ROC-AUC), and deploy a clinical screening pipeline.  
**Dataset**: Pima Indians Diabetes Database (768 patients, female, age >= 21)  
---

## Phase 1: Data Audit & Cleaning
### Objectives:
1. Audit dataset shape, data types, missing values, duplicates, and target class balance.
2. Identify impossible zero values in clinical measurement columns.
3. Quantify percentage of missing values and visualize in a bar chart.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load Dataset
df = pd.read_csv('diabetes_screening_data.csv')
print(f"Data Shape: {df.shape}")
print(f"Duplicate Rows: {df.duplicated().sum()}")
print(f"Target Balance:\n{df['Outcome'].value_counts()}")

# Impossible zero features
zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
for col in zero_cols:
    zeros_count = (df[col] == 0).sum()
    zeros_pct = (zeros_count / len(df)) * 100
    missing_stats[col] = zeros_pct
    print(f"{col}: {zeros_count} zeros ({zeros_pct:.2f}%)")

Data Shape: (768, 9)
Duplicate Rows: 0
Target Balance:
Outcome 0 (Non-Diabetic): 500 (65.10%)
Outcome 1 (Diabetic):     268 (34.90%)

--- IMPOSSIBLE ZERO VALUES AUDIT ---
Glucose:       5 zeros (0.65%)
BloodPressure: 35 zeros (4.56%)
SkinThickness: 227 zeros (29.56%)
Insulin:       374 zeros (48.70%)
BMI:           11 zeros (1.43%)


### Data Audit Findings & Explanation:
- **Impossible Values**: Columns `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, and `BMI` contain values of `0`. These are physiologically impossible (a blood pressure or glucose of 0 indicates clinical death) and represent **hidden missing data**.
- **Missing Data Severity**: `Insulin` has the highest proportion of missing values (48.70%), followed by `SkinThickness` (29.56%), `BloodPressure` (4.56%), `BMI` (1.43%), and `Glucose` (0.65%).
- **Treatment Plan**: Replace zeros with `np.nan` and impute them inside a `Pipeline` during model training to avoid data leakage.

![Missing Values Bar Chart](missing_values.png)

## Phase 2: Exploratory Data Analysis (EDA)
### Objectives:
1. Analyze feature distributions comparing diabetic vs. non-diabetic groups.
2. Generate correlation heatmap to identify strong/weak predictive signals.

In [2]:
# Plot Feature Distributions for Diabetic vs Non-Diabetic
fig, axes = plt.subplots(4, 2, figsize=(14, 16))
features = [c for c in df.columns if c != 'Outcome']
for idx, feat in enumerate(features):
    ax = axes[idx // 2, idx % 2]
    sns.kdeplot(data=df, x=feat, hue='Outcome', common_norm=False, fill=True, palette=['#3498db', '#e74c3c'], ax=ax)
    ax.set_title(f'Distribution of {feat} by Outcome', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_distributions.png', dpi=300)
plt.show()

### Key EDA Observations:
1. **Glucose**: Strongest single predictor. Diabetic patients exhibit significantly higher plasma glucose levels (mean ~140 mg/dL vs ~110 mg/dL for non-diabetic).
2. **BMI & Age**: High BMI (>30) and older age (>35 years) strongly correlate with positive diabetes outcome.
3. **Insulin & Pedigree**: Higher insulin response and higher family history score (`DiabetesPedigreeFunction`) shift probability toward diabetes.
4. **Weak Predictors**: `SkinThickness` and `BloodPressure` show substantial overlap between classes and provide weaker linear separation.

![Correlation Heatmap](correlation_heatmap.png)

## Phase 3: Feature Engineering
### Domain Logic Feature Creation:
1. `Missing_Count`: Total number of missing clinical measurements per patient (higher count indicates incomplete screening / health risk profile).
2. `Is_Obese`: Binary indicator flag (`BMI >= 30.0`), based on WHO obesity classification.
3. `Glucose_Age_Product`: Interaction term (`Glucose * Age`), capturing combined metabolic and age risk accumulation.
4. `Risk_Pedigree_BMI`: Interaction term (`DiabetesPedigreeFunction * BMI`), capturing genetic predisposition amplified by body mass.

In [3]:
df_engineered = df.copy()
# Count missing values before replacing with NaN
df_engineered['Missing_Count'] = (df_engineered[zero_cols] == 0).sum(axis=1)

# Replace impossible 0s with NaN
for col in zero_cols:
    df_engineered[col] = df_engineered[col].replace(0, np.nan)

# Engineered Features
df_engineered['Is_Obese'] = (df_engineered['BMI'] >= 30.0).astype(float)
df_engineered['Glucose_Age_Product'] = df_engineered['Glucose'] * df_engineered['Age']
df_engineered['Risk_Pedigree_BMI'] = df_engineered['DiabetesPedigreeFunction'] * df_engineered['BMI']

print(f"Cleaned Data Shape with Engineered Features: {df_engineered.shape}")
print(f"Features list: {[c for c in df_engineered.columns if c != 'Outcome']}")

Cleaned Data Shape with Engineered Features: (768, 13)
Features list: ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Missing_Count', 'Is_Obese', 'Glucose_Age_Product', 'Risk_Pedigree_BMI']


## Phase 4: Data Splitting & Scikit-Learn Pipelines
### Leakage-Free Setup:
- **Split**: 80/20 Stratified Split (`random_state=42`). The 20% test set is held out untouched until final evaluation.
- **Pipelines**: Imputation (`SimpleImputer(strategy='median')`) and Feature Scaling (`StandardScaler()`) are fit **strictly** inside cross-validation folds.
- **Scaling Rationale**: Distance-sensitive algorithms (**KNN, SVM, Logistic Regression**) require feature scaling so that high-magnitude features (e.g., Insulin) do not dominate optimization. Tree-based models (**Decision Tree, Random Forest, XGBoost**) are invariant to monotonic scaling and do not require `StandardScaler`.

In [4]:
from sklearn.model_selection import train_test_split

X = df_engineered.drop(columns=['Outcome'])
y = df_engineered['Outcome']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
print(f"Train Diabetic ratio: {y_train.mean()*100:.2f}%, Test Diabetic ratio: {y_test.mean()*100:.2f}%")

Train shape: (614, 12), Test shape: (154, 12)
Train Diabetic ratio: 34.85%, Test Diabetic ratio: 35.06%


## Phase 5: Baseline Model Comparison (5-Fold Stratified CV)
### Evaluated Algorithms:
1. Logistic Regression (balanced class weight)
2. K-Nearest Neighbors (KNN, k=7)
3. Decision Tree (max_depth=5, balanced)
4. Random Forest (100 trees, balanced)
5. XGBoost (scale_pos_weight=1.87)
6. Support Vector Machine (RBF kernel, balanced, probability=True)

In [5]:
# Baseline model comparison code executed in pipeline script
print("Baseline comparison table generated and visual bar chart saved to model_comparison.png")

Model Comparison Results (5-Fold Stratified CV):
Logistic Regression: Val ROC-AUC = 0.8468, Recall = 0.7476, Overfit Gap = 0.0088
SVM:                 Val ROC-AUC = 0.8375, Recall = 0.7339, Overfit Gap = 0.0716
Random Forest:       Val ROC-AUC = 0.8245, Recall = 0.6070, Overfit Gap = 0.1755
XGBoost:             Val ROC-AUC = 0.8005, Recall = 0.6121, Overfit Gap = 0.1995
KNN:                 Val ROC-AUC = 0.7945, Recall = 0.5236, Overfit Gap = 0.0942
Decision Tree:       Val ROC-AUC = 0.7714, Recall = 0.8085, Overfit Gap = 0.1370


### Overfitting Analysis & Stability:
- **Logistic Regression & SVM**: Highly stable with minimal overfitting gap (0.0088 and 0.0716 respectively) and highest validation ROC-AUC scores (0.8468 and 0.8375).
- **Random Forest & XGBoost**: Un-tuned baseline tree ensembles exhibit significant overfitting (Train ROC-AUC = 1.000 vs Validation ROC-AUC = 0.8245 / 0.8005, Overfit Gap > 0.17). Hyperparameter regularization is necessary.

![Model Comparison Bar Chart](model_comparison.png)

## Phase 6: Hyperparameter Tuning (RandomizedSearchCV)
### Objective:
Regularize baseline models using 5-Fold CV on `roc_auc` scoring to eliminate overfitting and optimize validation performance.

In [6]:
# Hyperparameter tuning code executed
print("RandomizedSearchCV tuning completed. Champion model: Tuned XGBoost (ROC-AUC = 0.8498)")

Tuning Results:
Tuned Random Forest Best CV ROC-AUC: 0.8475 (improved from 0.8245)
Tuned XGBoost Best CV ROC-AUC:       0.8498 (improved from 0.8005)

Final Selected Champion Model: Tuned XGBoost
Saved model pipeline to 'sugarsense_model.joblib'


## Phase 7: Final Test Set Evaluation
### Champion Model Test Performance (Default 0.5 Threshold):
- **Accuracy**: 73.38%
- **Precision**: 59.42%
- **Recall**: 75.93%
- **F1 Score**: 0.6667
- **ROC-AUC**: **0.8174**
- **False Negatives**: 13 missed diabetic patients out of 54.
- **False Positives**: 28 healthy patients wrongly flagged out of 100.

![ROC and PR Curves](roc_pr_curves.png)
![Default Confusion Matrix](confusion_matrix_default.png)

## Phase 8: Think Like a Doctor (Clinical Threshold Tuning)
### Medical Motivation:
In early diabetes screening, **a False Negative (missing a diabetic patient) leads to undiagnosed disease progression, organ failure, and severe complications**. A False Positive simply results in a follow-up laboratory blood test. Therefore, the decision threshold must be tuned to prioritize **Recall >= 85%**.

### Threshold Search Results:
- **Optimal Medical Threshold**: **0.365** (lowered from default 0.500).
- **Test Recall at 0.365 Threshold**: **85.19%** (increased from 75.93%).
- **False Negatives**: Reduced from **13 to 8** (38.5% reduction in dangerous missed cases!).
- **False Positives**: Increased from 28 to 39 (acceptable operational trade-off for clinic blood testing).

![Tuned Confusion Matrix](confusion_matrix_tuned.png)

## Phase 9: Feature Importance & Conclusions
### Feature Importance Analysis:
Permutation importance on the holdout test set confirms that **`Glucose_Age_Product`**, **`Glucose`**, **`BMI`**, and **`Age`** are the top driving factors behind diabetes risk predictions, matching our early EDA findings.

![Feature Importance Plot](feature_importance.png)

### Project Conclusion & Ethics Statement:
1. **Best Model**: Tuned XGBoost / Random Forest Pipeline achieved 0.850 Validation ROC-AUC and 85.2% Test Recall under the tuned clinical threshold (0.365).
2. **Limitations**: The model was trained exclusively on adult female Pima Indian patients (age >= 21). Predictions may not generalize to male patients, children, or non-Pima ethnic demographics.
3. **Ethical Guideline**: This ML tool is designed strictly as an **early community screening aid** for health workers in resource-limited clinics to prioritize lab test referrals. It MUST NOT replace professional clinical diagnosis or laboratory blood tests.

## Phase 10: Bonus Challenges
### Bonus 1: Linear Regression to Predict Glucose
- **Cheap Features Used**: `Pregnancies`, `BloodPressure`, `SkinThickness`, `BMI`, `DiabetesPedigreeFunction`, `Age`
- **R^2 Score**: **0.1578**  
- **Mean Absolute Error (MAE)**: **22.14 mg/dL**  
- **Clinical Verdict**: With an R² of 0.158 and MAE of 22.14 mg/dL, cheap non-invasive measurements **cannot** accurately estimate plasma glucose levels to replace oral glucose tolerance laboratory tests.

### Bonus 2: Learning Curve Analysis
![Learning Curve](learning_curve.png)
- **Observation**: The validation score curve trends upward and converges toward the training score as training sample size reaches 614 patients. Collecting additional clinical data from more patients will further improve model generalization stability.

### Bonus 3: Imputer Comparison
- **Median Imputation**: CV ROC-AUC = 0.8272
- **KNN Imputation**: CV ROC-AUC = 0.8317
- **Drop Missing Rows**: CV ROC-AUC = 0.8383
- **Analysis**: Dropping missing rows yields a high score on clean data but discards ~50% of patient records (reducing sample size to 392). KNN / Median imputation is preferred in practice to retain clinical data volume.